# 01 — A visual atlas from sky to Galactic phase space

**Question.** How do a star's direction, distance and motion become $R,\phi,Z,V_R,V_\phi,V_Z,L_Z$? Follow the same eight **TOY / SCHEMATIC** stars through the maps; then watch one real DESI/Gaia star go through Astropy.

The galaxy backgrounds provide astronomical context, not positional data. The only real stars here are the nearby 256-row public **DESI DR1 MWS Iron SV2 bright / Gaia DR3** teaching sample. They are not Lambert's unpublished DESI DR2 sample and cannot measure its outer-disc structure.

> **Reading key.** Gold circle = Sun; black diamond = Galactic Centre (GC); lettered coloured circles = the same toy stars. Blue arrows = outward; vermilion = disc-positive rotation; green = north/up. Angles and distances carry units throughout.

In [ ]:
from pathlib import Path
import json

import astropy
import astropy.units as u
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Circle, Rectangle
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import QTable
from mw_plot import MWFaceOn, MWSkyMap

from lambert_lab.atlas import TOY_COLORS, faceon_display_xy, sky_display_longitude, toy_atlas
from lambert_lab.coordinates import (
    GALCEN_DISTANCE, GALCEN_V_SUN, Z_SUN, cylindrical_velocity_components,
    disc_azimuth, galactocentric_frame, transform_observables,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sample = QTable.read(ROOT / 'data/derived/public_demo_sample.ecsv')
provenance = json.loads((ROOT / 'data/derived/public_demo_sample.provenance.json').read_text())
frame = galactocentric_frame()
toy = toy_atlas()
sun_x = SkyCoord(l=180*u.deg, b=0*u.deg, distance=0*u.kpc, frame='galactic').transform_to(frame).x.to_value(u.kpc)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
print(f'Astropy {astropy.__version__}; {len(sample)} real teaching stars; 8 TOY stars')
print(f"DESI source SHA-256: {sample.meta['source_sha256']}")
sample[['targetid', 'ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity', 'distance']][:4]

## 1 · Coordinate Rosetta stone

| What the observer has | What changes | What it tells us |
|---|---|---|
| ICRS $\alpha,\delta,d,\mu_{\alpha *},\mu_\delta,v_{\rm los}$ | sky direction, depth and velocity measured from the Sun | input measurements or a distance estimate |
| Galactic $l,b,d$ | rotate the **direction** to the Milky Way plane | where to look |
| Galactocentric $X,Y,Z$ | move the origin to the GC, with the adopted Sun position | where the star is in the Galaxy |
| Cylindrical $R,\phi,Z$ | replace $X,Y$ by radius and disc-positive angle | which annulus and height |
| $V_R,V_\phi,V_Z$ | resolve the **full** velocity in the local, moving basis | outward, with rotation, northward |
| $L_Z=R V_\phi$ | combine radius and disc-positive rotation | a useful phase-space axis |

**Watch the same letter across panels.** F and G share $(l,b)$ but differ in $d$: they occupy one sky pixel yet different Galactocentric locations.

## 2 · The sky is a map of directions

**Look:** $l$ runs along the Galactic plane; $b$ climbs toward the North Galactic Pole. GC is $l=0^\circ$, anticentre is $l=180^\circ$ at both map edges. The outlined halves are one continuous approximate Lambert window, $150<l<220^\circ$, $20<b<40^\circ$; the split is a map seam, not two selections. **Use:** this is the observational footprint before adding distances.

`mw-plot`'s bundled optical Gaia image is offline. Its horizontal display coordinate is $-l$ wrapped to $[-180,180]$; plotting methods accept **ICRS RA/Dec**, then convert internally. Image: **ESA/Gaia/DPAC**, as reported by `MWSkyMap.citation`. Background brightness is illustrative and is not this sample's star count.

In [ ]:
mw_sky = MWSkyMap(projection='equirectangular', background='optical', grid='galactic')
fig, ax = plt.subplots(figsize=(13, 6.3))
mw_sky.transform(ax)
mw_sky.imalpha = 0.8
ax.axhline(0, color='white', lw=1.3, alpha=.8, zorder=4)
for left, width in [(-180, 30), (140, 40)]:
    ax.add_patch(Rectangle((left, 20), width, 20, fill=False, ec='#ffe082', lw=2.8, zorder=7))
ax.scatter([0, -180, 180], [0, 0, 0], marker='D', s=[95, 65, 65],
           c=['#fff4e6']*3, edgecolor='black', zorder=8)
ax.text(4, 7, 'GC  $l=0°$', color='white', weight='bold', zorder=9)
ax.text(-175, -12, 'anticentre', color='white', zorder=9)
ax.text(145, -12, 'anticentre', color='white', zorder=9)
for row in toy:
    x = sky_display_longitude(row['l'])
    y = row['b'].to_value(u.deg)
    ax.scatter(x, y, s=65, c=row['color'], edgecolor='white', lw=1.2, zorder=9)
    if row['name'].split()[0] not in ('F','G'):
        ax.annotate(row['name'].split()[0], (x, y), xytext=(4, 5), textcoords='offset points',
                    color='white', weight='bold', fontsize=9, zorder=10)
ax.scatter(sky_display_longitude(toy['l'][5]),toy['b'][5].to_value(u.deg),
           s=150,facecolors='none',edgecolors=toy['color'][5],lw=2,zorder=10)
ax.text(-165,31,'F/G · same sky direction',color='white',fontsize=9,zorder=10)
ax.set(xlim=(-180, 180), ylim=(-72, 72), xlabel=r'display longitude $-l$ [deg]',
       ylabel=r'Galactic latitude $b$ [deg]', title='The Milky Way from the Sun · direction only')
ax.set_xticks([-180,-120,-60,0,60,120,180], ['180','120','60','0','300','240','180'])
ax.text(.99, .02, f'Background: {mw_sky.citation}', transform=ax.transAxes,
        color='white', ha='right', fontsize=8, bbox=dict(facecolor='black', alpha=.5, edgecolor='none'))
fig.tight_layout(); plt.show()

## 3 · Direction is not depth

**Look:** rays fix $(l,b)$; distance $d$ picks a point along one ray. F and G overlap on the sky, then separate in depth. **Use:** $d$ is measured from the Sun; Galactocentric $R$ is measured from the GC. They are generally unequal, even along the anticentre because the Sun already sits about 8.3 kpc from the GC.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax = axes[0]
for theta, label in [(0, r'$b=0°$'), (28, r'$b=28°$')]:
    t = np.deg2rad(theta)
    ax.plot([0, 12*np.cos(t)], [0, 12*np.sin(t)], color='#8b9da9', lw=1.7)
    ax.text(6.5*np.cos(t), 6.5*np.sin(t)+.3, label)
for j in [5,6]:
    d = toy['d'][j].to_value(u.kpc); t=np.deg2rad(28)
    ax.scatter(d*np.cos(t), d*np.sin(t), s=150, c=toy['color'][j], edgecolor='black', zorder=4)
    ax.text(d*np.cos(t)+.2, d*np.sin(t)+.25, toy['name'][j])
ax.scatter(0,0,s=180,c='#f2c14e',edgecolor='black',zorder=5); ax.text(.3,-.45,'Sun')
ax.set(xlim=(-.8,12),ylim=(-.7,6),xlabel='heliocentric planar distance [kpc]',ylabel='height [kpc]',title='Side view: same direction, different depths')
ax.set_aspect('equal')
ax=axes[1]
ax.scatter(0,0,s=150,marker='D',c='#343a40'); ax.text(.2,.3,'GC')
ax.scatter(sun_x,0,s=170,c='#f2c14e',edgecolor='black'); ax.text(sun_x,.3,'Sun')
for j in [5,6]:
    ax.plot([sun_x,toy['x'][j].to_value(u.kpc)],[0,toy['y'][j].to_value(u.kpc)],color=toy['color'][j],alpha=.7)
    ax.scatter(toy['x'][j].to_value(u.kpc),toy['y'][j].to_value(u.kpc),s=110,c=toy['color'][j],edgecolor='black')
    ax.annotate(toy['name'][j].split()[0],(toy['x'][j].to_value(u.kpc),toy['y'][j].to_value(u.kpc)),xytext=(5,5),textcoords='offset points')
ax.set(xlim=(-19,2),ylim=(-3,4),xlabel='project $X$ [kpc]',ylabel='project $Y$ [kpc]',title='Top view: $R$ starts at the GC')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 4 · Face-on context: where the rays land

**Look:** the Sun sits on the right of this artist's Milky Way; the anticentre ray points farther right. The same eight toy stars now have depth. **Use:** a sky patch becomes a curved region of the disc once distances are assigned.

This `mw-plot` image uses display coordinates $(x_{\rm display},y_{\rm display})=(-X_{\rm project},Y_{\rm project})$. The reflection places our negative-$X$ Sun at positive display x; all quantitative work below stays in project $X,Y$. We pass Lambert's $R_0=8.277$ kpc explicitly rather than the package's 8.125 kpc default. The artwork is schematic, not a calibrated density map. Image: **NASA/JPL-Caltech/R. Hurt (SSC/Caltech)**, from `MWFaceOn.citation`.

In [ ]:
mw_face = MWFaceOn(coord='galactocentric', r0=GALCEN_DISTANCE, radius=23*u.kpc)
fig, ax = plt.subplots(figsize=(8.2, 8.2))
mw_face.transform(ax)
sx = -sun_x
ax.scatter(0,0,s=150,c='#30343b',marker='D',edgecolor='white',zorder=6)
ax.scatter(sx,0,s=230,c='#f2c14e',edgecolor='black',zorder=7)
ax.text(.5,-1.2,'GC',color='white',weight='bold'); ax.text(sx+.5,-1.2,'Sun',color='white',weight='bold')
for l,c in [(0,'#fafafa'),(90,'#9adbe8'),(180,'#ffe082'),(270,'#e8a5ce')]:
    ray=SkyCoord(l=l*u.deg,b=0*u.deg,distance=np.array([0,13])*u.kpc,frame='galactic').transform_to(frame)
    xx,yy=faceon_display_xy(ray.x,ray.y)
    ax.plot(xx,yy,color=c,lw=1.7,ls='--',alpha=.9,zorder=5)
    ax.text(xx[-1],yy[-1],f'$l={l}°$',color=c,fontsize=9,ha='center',va='bottom')
for row in toy:
    x,y=faceon_display_xy(row['x'],row['y'])
    ax.scatter(x,y,s=95,c=row['color'],edgecolor='white',lw=1,zorder=8)
    label=row['name'].split()[0]
    offset={'B':(5,-14),'F':(5,8),'G':(5,4)}.get(label,(5,5))
    ax.annotate(label,(x,y),xytext=offset,textcoords='offset points',color='white',weight='bold',zorder=9)
ax.set(xlim=(-20,23),ylim=(-20,20),xlabel='illustration display x [kpc]',ylabel='illustration display y [kpc]',title='Face-on Milky Way · TOY sightlines and stars')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 5 · Scientific Cartesian views: the Sun is at negative $X$

**Look:** $X,Y,Z$ are signed distances from the GC. In this Astropy/project frame, the Sun is on the **negative $X$** side, $+Y$ is approximately $l=90^\circ$, and $+Z$ points north. The face-on artwork above deliberately reflected only the display x axis. **Use:** these are the quantitative axes of the transform and the starting point for $R=\sqrt{X^2+Y^2}$.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12.5,5.2))
for ax,horizontal,vertical,labels in [(axes[0],'x','y',('X','Y')),(axes[1],'x','z',('X','Z'))]:
    ax.axhline(0,color='#cbd5da',lw=1); ax.axvline(0,color='#cbd5da',lw=1)
    ax.scatter(0,0,s=115,marker='D',c='#30343b',zorder=5)
    ax.scatter(sun_x, Z_SUN.to_value(u.kpc) if vertical=='z' else 0,s=155,c='#f2c14e',edgecolor='black',zorder=5)
    for row in toy:
        x=row[horizontal].to_value(u.kpc); y=row[vertical].to_value(u.kpc)
        ax.scatter(x,y,s=100,c=row['color'],edgecolor='white',lw=.8,zorder=6)
        ax.annotate(row['name'].split()[0],(x,y),xytext=(5,4),textcoords='offset points',fontsize=9)
    ax.set(xlabel=f'{labels[0]} [kpc]',ylabel=f'{labels[1]} [kpc]',title=f'Galactocentric {labels[0]}–{labels[1]}')
    ax.set_aspect('equal'); ax.grid(alpha=.14)
axes[0].set(xlim=(-23,5),ylim=(-15,15)); axes[1].set(xlim=(-23,5),ylim=(-5,9))
fig.tight_layout(); plt.show()

## 6 · Signature linked view: one cast, three coordinate systems

**Look:** F and G are coincident in $l,b$, separate in $X,Y$, and separate again in $R,Z$. The outer anticentre H is far from the GC and above the plane. **Use:** Lambert's sky selection is a direction cut; phase-space plots require distances and velocities too. All points in this figure are **TOY / SCHEMATIC**.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.8),gridspec_kw={'width_ratios':[1,1.15,1.1]})
axes[0].add_patch(Rectangle((150,20),70,20,fill=False,ec='#d49a00',lw=2))
for row in toy:
    views=[(row['l'].to_value(u.deg),row['b'].to_value(u.deg)),
           (row['x'].to_value(u.kpc),row['y'].to_value(u.kpc)),
           (row['R'].to_value(u.kpc),row['z'].to_value(u.kpc))]
    for k,(x,y) in enumerate(views):
        axes[k].scatter(x,y,s=110,c=row['color'],edgecolor='black',lw=.5,zorder=4)
        if k or row['name'][0] not in ('F','G'):
            axes[k].annotate(row['name'].split()[0],(x,y),xytext=(5,5),textcoords='offset points',fontsize=9)
axes[0].scatter(175,28,s=170,facecolors='none',edgecolors=toy['color'][5],lw=2)
axes[0].text(184,31,'F/G',fontsize=9)
axes[0].set(xlim=(-10,360),ylim=(-12,57),xlabel='$l$ [deg]',ylabel='$b$ [deg]',title='sky · direction')
axes[1].scatter([0,sun_x],[0,0],marker='D',s=65,c=['#30343b','#f2c14e'],zorder=5)
axes[1].set(xlim=(-23,5),ylim=(-10,10),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='disc plane · position')
axes[2].set(xlim=(0,24),ylim=(-2,9),xlabel='$R$ [kpc]',ylabel='$Z$ [kpc]',title='cylindrical · radius and height')
for ax in axes[1:]: ax.set_aspect('equal'); ax.grid(alpha=.15)
fig.suptitle('Same eight TOY stars · identical labels and colours',fontsize=15)
fig.tight_layout(); plt.show()

## 7 · From $X,Y,Z$ to a local cylindrical address

**Look:** circles share $R$; spokes share project-defined disc-positive $\phi=\operatorname{atan2}(Y,-X)$. Zero is on the GC→Sun ray and positive $\phi$ points toward $+Y$ near the Sun. $Z$ is the height above the plane. **Use:** $R$ and $\phi$ label an annulus and a place around it; Lambert later studies radius and velocity together.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11.5,5))
ax=axes[0]
for radius in [5,10,15,20]:
    ax.add_patch(Circle((0,0),radius,fill=False,ec='#9db5bd',lw=1,ls='--'))
    ax.text(-radius,0.5,f'{radius} kpc',fontsize=8,color='#546b75')
for phi in [-60,-30,0,30,60]:
    t=np.deg2rad(phi)
    ax.plot([0,-21*np.cos(t)],[0,21*np.sin(t)],color='#c7d1d5',lw=1)
    ax.text(-21*np.cos(t),21*np.sin(t),f'{phi}°',fontsize=8)
ax.scatter(0,0,marker='D',s=100,c='#30343b'); ax.scatter(sun_x,0,s=140,c='#f2c14e',edgecolor='black')
for row in toy:
    ax.scatter(row['x'].to_value(u.kpc),row['y'].to_value(u.kpc),c=row['color'],s=70,zorder=5)
ax.set(xlim=(-23,7),ylim=(-20,20),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title=r'Constant $R$ rings and $\phi$ spokes')
ax.set_aspect('equal')
ax=axes[1]
for row in toy:
    ax.scatter(row['R'].to_value(u.kpc),row['z'].to_value(u.kpc),c=row['color'],s=100,edgecolor='black',lw=.5)
    ax.annotate(row['name'].split()[0],(row['R'].to_value(u.kpc),row['z'].to_value(u.kpc)),xytext=(5,4),textcoords='offset points',fontsize=9)
ax.axhline(0,color='#8c9da5'); ax.set(xlim=(0,24),ylim=(-1,9),xlabel='$R$ [kpc]',ylabel='$Z$ [kpc]',title='Height is retained')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 8 · The velocity basis turns with position

**Look:** $\mathbf e_R$ always points away from the GC; $\mathbf e_\phi$ follows positive disc rotation; $\mathbf e_Z$ points out of this page everywhere. The arrows rotate around the disc. **Use:** a single fixed Cartesian velocity direction can be outward at one position and tangential at another.

In [ ]:
fig,ax=plt.subplots(figsize=(7.2,6.2))
for radius in [7,14]: ax.add_patch(Circle((0,0),radius,fill=False,ec='#d5e0e3',ls='--'))
for phi in np.deg2rad([0,60,120,180,240,300]):
    x,y=-12*np.cos(phi),12*np.sin(phi)
    er=np.array([x,y])/12
    ep=np.array([er[1],-er[0]])
    ax.arrow(x,y,2.5*er[0],2.5*er[1],color='#0072b2',width=.13,head_width=.65,length_includes_head=True)
    ax.arrow(x,y,2.5*ep[0],2.5*ep[1],color='#d55e00',width=.13,head_width=.65,length_includes_head=True)
    ax.scatter(x,y,s=55,c='#30343b')
ax.scatter(0,0,s=110,marker='D',c='#30343b'); ax.scatter(sun_x,0,s=135,c='#f2c14e',edgecolor='black')
ax.plot([],[],color='#0072b2',lw=3,label=r'outward $\mathbf{e}_R$')
ax.plot([],[],color='#d55e00',lw=3,label=r'disc rotation $\mathbf{e}_\phi$')
ax.scatter([],[],s=55,c='#009e73',label=r'$\mathbf{e}_Z$: toward you / north')
ax.legend(loc='lower right',framealpha=.95,fontsize=9)
ax.set(xlim=(-17,17),ylim=(-17,17),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='Local cylindrical basis · TOY sites')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 9 · Three signs, three physical motions

**Look:** near the Sun on negative $X$, outward motion points toward more negative $X$; positive disc rotation points toward $+Y$; positive $V_Z$ points north. These are components of one 3D velocity. **Use:** Lambert's $V_R$ and $V_Z$ maps distinguish radial and vertical streaming; $V_\phi$ enters its $R$–$V_\phi$ projection.

The project helper follows Lambert's positive disc-rotation $V_\phi$ and defines $L_Z=R V_\phi$. This differs in sign from the angular direction of Astropy's `atan2(Y,X)`; the plotted project $\phi$ is `atan2(Y,-X)`. Lambert does not specify a plotted azimuth angle, so this notebook claims agreement only for the documented physical velocity signs.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11.4,4.3))
ax=axes[0]; x0=sun_x-2
ax.scatter(0,0,s=120,marker='D',c='#30343b'); ax.scatter(sun_x,0,s=130,c='#f2c14e',edgecolor='black')
ax.scatter(x0,0,s=70,c='#30343b')
ax.annotate('',(x0-3,0),(x0,0),arrowprops=dict(arrowstyle='-|>',lw=3,color='#0072b2'))
ax.annotate('',(x0,3),(x0,0),arrowprops=dict(arrowstyle='-|>',lw=3,color='#d55e00'))
ax.text(x0-5,-1.1,r'$V_R>0$',color='#0072b2'); ax.text(x0+.4,2,r'$V_\phi>0$',color='#d55e00')
ax.set(xlim=(-16,3),ylim=(-3,5),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='Disc plane · outward and rotation')
ax.set_aspect('equal')
ax=axes[1]; ax.axhline(0,color='#b3c2c8'); ax.scatter(0,0,s=90,c='#30343b')
ax.annotate('',(0,3),(0,0),arrowprops=dict(arrowstyle='-|>',lw=3,color='#009e73'))
ax.text(.35,2,r'$V_Z>0$ · North Galactic Pole',color='#009e73')
ax.set(xlim=(-2,8),ylim=(-1,4),xlabel='local horizontal direction',ylabel='$Z$ [kpc]',title='Side view · vertical sign')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 10 · A radial velocity from the observer is not $V_R$

**Look:** $v_{\rm los}$ lies on the Sun→star ray; proper motion measures angular displacement across that ray, which becomes transverse speed after using $d$. Galactocentric $V_R$ points along GC→star after changing origin and subtracting the Sun's motion. **Use:** a sightline velocity by itself cannot tell whether a star moves outward from the GC.

In [ ]:
fig,ax=plt.subplots(figsize=(8.3,5.3))
sun=np.array([sun_x,0]); star=np.array([-14.,5.]); ray=(star-sun)/np.linalg.norm(star-sun)
perp=np.array([-ray[1],ray[0]]); er=star/np.linalg.norm(star)
ax.plot([sun[0],star[0]],[sun[1],star[1]],color='#a8b8c0',lw=2,ls='--')
ax.plot([0,star[0]],[0,star[1]],color='#c6d2d6',lw=1.5)
ax.scatter(0,0,s=125,marker='D',c='#30343b'); ax.text(.3,-.4,'GC')
ax.scatter(*sun,s=155,c='#f2c14e',edgecolor='black'); ax.text(sun[0]+.3,-.5,'Sun')
ax.scatter(*star,s=100,c='#0072b2'); ax.text(star[0]-.3,star[1]+.4,'TOY star')
for vec,color,label,offset in [(ray,'#7b3294',r'$v_{\rm los}$',(0,.3)),(perp,'#009e73','proper motion × $d$',(-5,.2)),(er,'#0072b2',r'Galactocentric $V_R$',(-6,-.5))]:
    end=star+3*vec
    ax.annotate('',end,star,arrowprops=dict(arrowstyle='-|>',lw=2.8,color=color))
    ax.text(end[0]+offset[0],end[1]+offset[1],label,color=color,fontsize=10)
ax.set(xlim=(-24,3),ylim=(-3,12),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='Two origins, different radial directions')
ax.set_aspect('equal'); fig.tight_layout(); plt.show()

## 11 · What DESI and Gaia actually measured

### Question
Which entries are measurements, and which quantities will we compute?

### Physical intuition
A position on the sky gives a direction, not a three-dimensional location. Gaia adds parallax and the two components of angular motion. DESI adds the motion along our line of sight. Together, these six phase-space coordinates allow a frame transformation.

- `ra`, `dec`: Gaia ICRS sky direction.
- `parallax`: the apparent annual displacement caused by Earth's orbit.
- `pmra`: $\mu_{\alpha *}=\dot{\alpha}\cos\delta$. The star's catalogue already includes the $\cos\delta$ factor; passing plain $\dot{\alpha}$ to Astropy would be wrong.
- `pmdec`: angular motion in declination.
- `radial_velocity`: DESI RVSpecFit line-of-sight velocity, positive when receding from the observer.

> **OBSERVABLE / MEASUREMENT:** the catalogue columns above and their uncertainties.  
> **COMPUTATIONAL OPERATION:** quality filtering, inverse parallax, and all coordinate transformations below.

### Why this tiny sample?

The builder retains successful stellar DESI spectra with clean fibre status and $\sigma(v_{\rm los})\leq5$ km s$^{-1}$, then requires a five-parameter Gaia solution, at least eight visibility periods, `RUWE < 1.4`, no Gaia duplicated-source flag, $\varpi>0.5$ mas, and $\varpi/\sigma_\varpi\geq20$. `RUWE < 1.4` is a conventional heuristic, not a universal truth about Gaia solutions. Repeated Gaia source IDs are ranked by smallest radial-velocity uncertainty, highest R-arm S/N, then lowest DESI `TARGETID`; this file had no surviving repeats. A seeded random draw limits the lesson to 256 stars.

> **Distance caveat.** This sample is deliberately nearby and has positive, high-S/N parallaxes, so we use the pedagogical approximation $d[\mathrm{kpc}]=1/\varpi[\mathrm{mas}]$. This is not a complete treatment of Gaia systematic errors or distance posteriors. No Gaia parallax zero-point correction is applied. Never extrapolate this shortcut to the distant outer disc: Lambert's MSTO analysis is a fundamentally different distance regime.

In [ ]:
assert np.all(sample['parallax'] > 0.5 * u.mas)
assert np.all(sample['parallax_over_error'] >= 20)
assert u.allclose(sample['distance'], (1 / sample['parallax'].to_value(u.mas)) * u.kpc)
print('Cut flow (remaining rows):')
for item in provenance['cut_flow']:
    print(f"{item['remaining']:5d}  {item['cut']}")

## 12 · Adopted frame and sign definitions

### Physical intuition before equations
The observer-centred Galactic frame replaces $(\alpha,\delta)$ with longitude $l$ and latitude $b$. A Galactocentric transformation then changes the origin to the Galactic centre and accounts for the Sun's specified Galactocentric position and velocity. This is why an observed line-of-sight velocity is not, by itself, $V_R$.

The installed Astropy convention is checked directly here (see its [Galactocentric documentation](https://docs.astropy.org/en/stable/coordinates/galactocentric.html)). Astropy's Galactocentric Cartesian frame is right-handed: $x$ points approximately from the Sun toward the Galactic centre, so the Sun is at **negative** $x$; $y$ points roughly toward $l=90^\circ$; $z$ points to the North Galactic Pole. We define

$$R=\sqrt{x^2+y^2},\qquad \phi=\operatorname{atan2}(y,-x),$$

so $\phi=0$ on the Galactic-centre-to-Sun ray and positive $\phi$ follows disc rotation near the Sun. Then

$$V_R=\frac{xv_x+yv_y}{R},\qquad V_\phi=\frac{yv_x-xv_y}{R},\qquad V_Z=v_z.$$

Thus $V_R>0$ means outward, $V_\phi>0$ means prograde/disc rotation, and $V_Z>0$ means north/up. This disc-positive $V_\phi$ is the negative of the velocity along increasing `atan2(y, x)` in Astropy's axes. We use the Lambert-compatible disc-positive quantity $L_Z=R V_\phi$; it is positive for ordinary disc rotation. Lambert states a right-handed Galactocentric transformation and uses positive disc $V_\phi$ and $L_Z$; the paper does not explicitly define its plotted azimuth angle, so only these physical velocity meanings are claimed to be reconciled.

## 13 · Predict one easy star before transforming

### Prediction exercise
Imagine a synthetic star with the same Galactocentric azimuth as the Sun–anticentre line, farther from the Galactic centre than the Sun and 1 kpc above the plane: $(x,y,z)=(-10,0,1)$ kpc in Astropy's frame. Give it $(v_x,v_y,v_z)=(-30,220,15)$ km s$^{-1}$.

Before running the next cell, predict the signs. At negative $x$, a negative $v_x$ moves farther from the centre, so $V_R$ should be positive. Positive $v_y$ is disc rotation there, so $V_\phi$ should be positive. Positive $v_z$ points north, so $V_Z$ should be positive. We first turn this understandable Galactocentric construction into mock sky observables, then ask Astropy to recover it.

In [ ]:
synthetic_gc = SkyCoord(
    x=-10*u.kpc, y=0*u.kpc, z=1*u.kpc,
    v_x=-30*u.km/u.s, v_y=220*u.km/u.s, v_z=15*u.km/u.s,
    frame=frame, representation_type='cartesian', differential_type='cartesian',
)
mock_observed = synthetic_gc.transform_to('icrs')
print('Mock observables:', mock_observed)
recovered = mock_observed.transform_to(frame)
syn_vr, syn_vphi, syn_vz = cylindrical_velocity_components(
    recovered.x.to_value(u.kpc), recovered.y.to_value(u.kpc),
    recovered.v_x.to_value(u.km/u.s), recovered.v_y.to_value(u.km/u.s),
    recovered.v_z.to_value(u.km/u.s),
)
print(f'Recovered: V_R={syn_vr:.1f}, V_phi={syn_vphi:.1f}, V_Z={syn_vz:.1f} km/s')
assert np.allclose([syn_vr, syn_vphi, syn_vz], [30, 220, 15], atol=1e-9)

## 14 · A real star, transparently

For one catalogue row we explicitly construct `SkyCoord`, rather than hiding the operation in a helper. Astropy interprets `pm_ra_cosdec` as $\mu_{\alpha *}$ and combines angular motion, distance, and line-of-sight velocity into a Cartesian velocity before changing the origin.

In [ ]:
star = sample[0]
one_sky = SkyCoord(
    ra=star['ra'], dec=star['dec'], distance=star['distance'],
    pm_ra_cosdec=star['pmra'], pm_dec=star['pmdec'],
    radial_velocity=star['radial_velocity'], frame='icrs',
)
print(f"ICRS: ra={one_sky.ra:.3f}, dec={one_sky.dec:.3f}, d={one_sky.distance:.3f}")
print(f"observed: mu_alpha*={one_sky.pm_ra_cosdec:.3f}, mu_delta={one_sky.pm_dec:.3f}, v_los={one_sky.radial_velocity:.2f}")
one_galactic = one_sky.galactic
one_gc = one_sky.transform_to(frame)
one_R = np.hypot(one_gc.x, one_gc.y)
one_phi = disc_azimuth(one_gc.x, one_gc.y)
one_vr, one_vphi, one_vz = cylindrical_velocity_components(
    one_gc.x.to_value(u.kpc), one_gc.y.to_value(u.kpc),
    one_gc.v_x.to_value(u.km/u.s), one_gc.v_y.to_value(u.km/u.s),
    one_gc.v_z.to_value(u.km/u.s),
)
one_lz = one_R * one_vphi * u.km/u.s
print(f"l={one_galactic.l:.2f}, b={one_galactic.b:.2f}")
print(f"(x,y,z)=({one_gc.x:.3f}, {one_gc.y:.3f}, {one_gc.z:.3f})")
print(f"(v_x,v_y,v_z)=({one_gc.v_x:.2f}, {one_gc.v_y:.2f}, {one_gc.v_z:.2f})")
print(f"R={one_R:.3f}, phi={one_phi:.2f}")
print(f"(V_R,V_phi,V_Z)=({one_vr:.2f}, {one_vphi:.2f}, {one_vz:.2f}) km/s")
print(f"L_Z={one_lz.to(u.kpc*u.km/u.s):.1f}")
round_trip = one_gc.transform_to('icrs')
assert one_sky.separation_3d(round_trip).to_value(u.pc) < 1e-8

## 15 · The same operation across the real sample

The helper below performs exactly the construction just shown and adds $l,b,x,y,z,R,\phi,V_R,V_\phi,V_Z,L_Z$ in memory. Those derived columns are intentionally absent from the committed ECSV so the notebook remains the place where the transformation happens.

In [ ]:
phase_space = transform_observables(sample)
assert phase_space['R'].unit == u.kpc
assert phase_space['V_phi'].unit == u.km/u.s
assert phase_space['L_Z'].unit == u.kpc*u.km/u.s
phase_space[['R', 'phi', 'z', 'V_R', 'V_phi', 'V_Z', 'L_Z']][:5]

In [ ]:
vr_values = phase_space['V_R'].to_value(u.km/u.s)
vmax = np.max(np.abs(vr_values))
vr_norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
fig, ax = plt.subplots(figsize=(7, 4.5))
points = ax.scatter(phase_space['R'].to_value(u.kpc), phase_space['V_phi'].to_value(u.km/u.s),
                    c=vr_values, cmap='coolwarm', norm=vr_norm, s=24, alpha=0.8)
ax.set(xlabel=r'$R$ [kpc]', ylabel=r'$V_\phi$ [km s$^{-1}$]',
       title=r'Nearby SV2 teaching stars ($V_R>0$: outward; $V_R<0$: inward)')
colorbar = fig.colorbar(points, ax=ax, label=r'$V_R$ [km s$^{-1}$]  ($>0$ outward; $<0$ inward)')
fig.tight_layout()
plt.show()

### What to notice

Most of this nearby, deliberately selected sample lies close to the solar radius; many stars have positive disc-rotation velocity, while $V_R$ spans both signs. This is a compact diagnostic of the transformed quantities, **not** a population measurement: SV2 targeting and our cuts are not a survey selection function.

> **DATA-SUPPORTED INFERENCE:** an individual star's signs describe its instantaneous motion under the adopted frame and distance estimate.  
> **MODEL-DEPENDENT INTERPRETATION:** explaining a multi-star ridge or wave requires dynamics and a controlled selection function; this sample does not establish such a structure.

## 16 · Why $R$–$V_\phi$ and $L_Z$ are useful

**Look:** at fixed positive $L_Z=R V_\phi$, $V_\phi=L_Z/R$ falls as $R$ grows. These are geometry guides, not detected ridges. **Use:** Lambert later presents density in this projection; interpreting a ridge requires its actual released products and a dynamical model, handled in Notebooks 02 and 04.

In [ ]:
fig,ax=plt.subplots(figsize=(8.2,4.8))
r=np.linspace(6,24,400)
for lz,c in [(1800,'#56b4e9'),(2600,'#009e73'),(3400,'#d55e00')]:
    ax.plot(r,lz/r,color=c,lw=2,label=fr'$L_Z={lz}$ kpc km s$^{{-1}}$')
ax.scatter(phase_space['R'].to_value(u.kpc),phase_space['V_phi'].to_value(u.km/u.s),
           s=16,c='#30343b',alpha=.35,label='real nearby SV2 teaching stars')
ax.set(xlim=(5.5,24),ylim=(80,430),xlabel='$R$ [kpc]',ylabel=r'$V_\phi$ [km s$^{-1}$]',title='Constant $L_Z$ curves · educational geometry')
ax.legend(fontsize=8,loc='upper right'); ax.grid(alpha=.17); fig.tight_layout(); plt.show()

## 17 · Try it yourself

Predict first: if the same star's measured line-of-sight velocity were 20 km s$^{-1}$ larger, would $V_R$, $V_\phi$, or both change? The answer depends on the sightline—the line-of-sight basis is generally not aligned with a Galactocentric cylindrical basis. Change `velocity_kick` and inspect the result.

In [ ]:
velocity_kick = 20 * u.km/u.s
experiment = sample[:1].copy()
baseline = transform_observables(experiment)
experiment['radial_velocity'] += velocity_kick
changed = transform_observables(experiment)
for component in ['V_R', 'V_phi', 'V_Z']:
    delta = (changed[component][0] - baseline[component][0]).to(u.km/u.s)
    print(f'Delta {component} = {delta:.2f}')

## 18 · Lambert geometry bridge: a rectangle on the sky, a curved volume in the disc

**Look:** a rectangular cut in $(l,b)$ does not become a rectangle in $(X,Y)$ or $(R,Z)$. The shown boundaries use **TOY distances 4–14 kpc** solely to demonstrate geometry; they are not Lambert's distance cuts, sources, ACS/MRi memberships, or a recovered survey volume. **Use:** selection geometry must be understood before reading an outer-disc map.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.6))
axes[0].add_patch(Rectangle((150,20),70,20,fc='#f2c14e',alpha=.25,ec='#a66b00',lw=2))
axes[0].set(xlim=(130,240),ylim=(5,55),xlabel='$l$ [deg]',ylabel='$b$ [deg]',title='angular rectangle')
lgrid=np.linspace(150,220,120)*u.deg
for b in [20,40]:
    for d in [4,14]:
        g=SkyCoord(l=lgrid,b=b*u.deg,distance=d*u.kpc,frame='galactic').transform_to(frame)
        axes[1].plot(g.x.to_value(u.kpc),g.y.to_value(u.kpc),color='#a66b00',lw=1.6)
        axes[2].plot(np.hypot(g.x,g.y).to_value(u.kpc),g.z.to_value(u.kpc),color='#a66b00',lw=1.6)
for l in [150,220]:
    for d in [4,14]:
        g=SkyCoord(l=l*u.deg,b=np.linspace(20,40,80)*u.deg,distance=d*u.kpc,frame='galactic').transform_to(frame)
        axes[1].plot(g.x.to_value(u.kpc),g.y.to_value(u.kpc),color='#a66b00',lw=1.2)
        axes[2].plot(np.hypot(g.x,g.y).to_value(u.kpc),g.z.to_value(u.kpc),color='#a66b00',lw=1.2)
axes[1].scatter(sun_x,0,s=100,c='#f2c14e',edgecolor='black'); axes[1].scatter(0,0,s=80,c='#30343b',marker='D')
axes[1].set(xlim=(-23,0),ylim=(-10,10),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='projected disc footprint')
axes[2].set(xlim=(7,24),ylim=(0,10),xlabel='$R$ [kpc]',ylabel='$Z$ [kpc]',title='radius–height footprint')
for ax in axes[1:]: ax.set_aspect('equal'); ax.grid(alpha=.15)
fig.suptitle('Approximate Lambert angular window + illustrative depth bounds · TOY',fontsize=13)
fig.tight_layout(); plt.show()

## 19 · The whole map on one slide

Read left to right: a direction plus distance locates a star; the local cylindrical basis resolves its velocity; $R$ and disc-positive $V_\phi$ combine into $L_Z$. The symbols trace **one TOY outer-disc star H**. The equations and signs follow the project frame; no Lambert DR2 stellar measurements are drawn here.

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(16,4.1))
h=toy[7]; hx=h['x'].to_value(u.kpc); hy=h['y'].to_value(u.kpc); hr=h['R'].to_value(u.kpc)
ax=axes[0]; ax.add_patch(Rectangle((150,20),70,20,fill=False,ec='#d49a00',lw=2))
ax.scatter(h['l'].to_value(u.deg),h['b'].to_value(u.deg),s=125,c=h['color'],edgecolor='black')
ax.set(xlim=(140,225),ylim=(10,48),xlabel='$l$ [deg]',ylabel='$b$ [deg]',title='1  sky direction')
ax=axes[1]; ax.scatter([0,sun_x],[0,0],s=[75,105],c=['#30343b','#f2c14e'],marker='o')
ax.plot([sun_x,hx],[0,hy],color='#889ba3',ls='--'); ax.scatter(hx,hy,s=115,c=h['color'],edgecolor='black')
ax.set(xlim=(-23,2),ylim=(-8,8),xlabel='$X$ [kpc]',ylabel='$Y$ [kpc]',title='2  position'); ax.set_aspect('equal')
ax=axes[2]; er=np.array([hx,hy])/hr; ep=np.array([er[1],-er[0]])
ax.scatter(0,0,s=115,c=h['color'],edgecolor='black')
ax.arrow(0,0,*er,color='#0072b2',head_width=.18,width=.04,length_includes_head=True)
ax.arrow(0,0,*ep,color='#d55e00',head_width=.18,width=.04,length_includes_head=True)
ax.text(-1.1,-1.2,r'$+V_R$',color='#0072b2'); ax.text(.2,-1.2,r'$+V_\phi$',color='#d55e00')
ax.set(xlim=(-1.4,1.4),ylim=(-1.4,1.4),xlabel='project $X$ direction',ylabel='project $Y$ direction',title='3  local basis'); ax.set_aspect('equal')
ax=axes[3]; rr=np.linspace(8,24,200); lz=hr*220
ax.plot(rr,lz/rr,color='#d55e00',lw=2); ax.scatter(hr,220,s=120,c=h['color'],edgecolor='black',zorder=5)
ax.set(xlim=(8,24),ylim=(120,430),xlabel='$R$ [kpc]',ylabel=r'$V_\phi$ [km s$^{-1}$]',title=r'4  $L_Z=R V_\phi$')
fig.suptitle('From a sky measurement to a phase-space address · TOY star H',fontsize=15)
fig.tight_layout(); plt.show()

## Where to go next

We now know what the signs mean and how $L_Z=R V_\phi$ is constructed. Lambert et al. use these Galactocentric quantities to study distant MSTO stars and released downstream maps/waves. That science does **not** follow from this nearby SV2 sample, and the inverse-parallax shortcut used here must not be carried into Lambert's distance regime. Later notebooks use Lambert's released figure products, not this catalogue, and must distinguish measured patterns from dynamical interpretation—including any proposed link to Sagittarius.